---
## Task 1 – New Engineered Feature: `study_efficiency`

**Justification:** `study_efficiency` = `studytime / (failures + 1)` combines how much a student studies with how many times they have failed. A student who studies a lot but still fails repeatedly has a very different academic profile from one who studies little but has no failures. This ratio may help the model distinguish at-risk students more effectively than either feature alone, particularly for the failing class.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

# Load dataset
df = pd.read_csv("student-mat (1).csv", sep=";")

# Target: pass if final grade G3 >= 10
df["pass"] = (df["G3"] >= 10).astype(int)

# Drop raw grade columns to avoid leakage
drop_cols = ["G1", "G2", "G3"]
target_col = "pass"

# Base feature matrix (no extra features yet)
X_base = df.drop(columns=drop_cols + [target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

def build_and_eval(X_tr, X_te, y_tr, y_te, label=""):
    cat_cols = X_tr.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ])
    pipe = Pipeline([
        ("preprocess", pre),
        ("rf", RandomForestClassifier(n_estimators=300, random_state=42,
                                      n_jobs=-1, class_weight="balanced"))
    ])
    pipe.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, pipe.predict(X_te))
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"  Accuracy: {acc:.4f}")
    print(classification_report(y_te, pipe.predict(X_te), zero_division=0))
    return pipe, acc

# Baseline
pipe_base, acc_base = build_and_eval(X_train, X_test, y_train, y_test, "Baseline")

---
## Task 1 – New Engineered Feature: `study_efficiency`

**Justification:** `study_efficiency` = `studytime / (failures + 1)` combines how much a student studies with how many times they have failed. A student who studies a lot but still fails repeatedly has a very different academic profile from one who studies little but has no failures. This ratio may help the model distinguish at-risk students more effectively than either feature alone, particularly for the failing class.

In [ ]:
df_t1 = df.drop(columns=drop_cols + [target_col]).copy()
df_t1["study_efficiency"] = df["studytime"] / (df["failures"] + 1)

Xtr1, Xte1, ytr1, yte1 = train_test_split(df_t1, y, test_size=0.2, random_state=42, stratify=y)
pipe1, acc1 = build_and_eval(Xtr1, Xte1, ytr1, yte1, "Task 1 – with study_efficiency")

print(f"Baseline accuracy       : {acc_base:.4f}")
print(f"+ study_efficiency acc  : {acc1:.4f}")

---
## Task 2 – Different Rule for `is_high_absences`

| Rule | Threshold |
|------|----------|
| Original | `absences > 5` |
| **New** | `absences > 10` |

The original threshold flags students with more than 5 absences. We try a stricter threshold of 10 to capture only severely absent students and discuss whether this changes model performance.

In [ ]:
# Original rule
df_orig = df.drop(columns=drop_cols + [target_col]).copy()
df_orig["is_high_absences"] = (df["absences"] > 5).astype(int)
Xtr_o, Xte_o, ytr_o, yte_o = train_test_split(df_orig, y, test_size=0.2, random_state=42, stratify=y)
_, acc_orig = build_and_eval(Xtr_o, Xte_o, ytr_o, yte_o, "Task 2 – is_high_absences (threshold > 5)")

# New rule
df_new = df.drop(columns=drop_cols + [target_col]).copy()
df_new["is_high_absences"] = (df["absences"] > 10).astype(int)
Xtr_n, Xte_n, ytr_n, yte_n = train_test_split(df_new, y, test_size=0.2, random_state=42, stratify=y)
_, acc_new = build_and_eval(Xtr_n, Xte_n, ytr_n, yte_n, "Task 2 – is_high_absences (threshold > 10)")

print(f"\nThreshold > 5   accuracy: {acc_orig:.4f}")
print(f"Threshold > 10  accuracy: {acc_new:.4f}")
print()
print("Discussion: a stricter threshold (> 10) labels fewer students as high-absence.")
print("If accuracy drops, the mild-absence group (6–10) contained useful signal.")
print("If accuracy holds, only severely absent students drive the prediction.")

---
## Task 3 – Vary the Pass Threshold for the Target

The pass/fail label depends on the grade threshold. We test thresholds `{10, 11, 12, 13}` and compare accuracy and feature importances.

In [ ]:
results_task3 = {}

for threshold in [10, 11, 12, 13]:
    y_t = (df["G3"] >= threshold).astype(int)
    X_t = df.drop(columns=drop_cols + [target_col])
    Xtr, Xte, ytr, yte = train_test_split(X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)

    cat_c = Xtr.select_dtypes(include=["object", "category"]).columns.tolist()
    num_c = Xtr.select_dtypes(include=[np.number]).columns.tolist()
    pre   = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_c),
                               ("num", "passthrough", num_c)])
    pipe  = Pipeline([("pre", pre),
                      ("rf",  RandomForestClassifier(n_estimators=300, random_state=42,
                                                     n_jobs=-1, class_weight="balanced"))])
    pipe.fit(Xtr, ytr)
    acc = accuracy_score(yte, pipe.predict(Xte))
    results_task3[threshold] = acc
    pass_rate = y_t.mean()
    print(f"  threshold={threshold}  pass_rate={pass_rate:.2f}  accuracy={acc:.4f}")

print()
print("Observation: raising the threshold reduces the pass rate, making the task harder.")
print("Accuracy may appear high for extreme thresholds simply due to class imbalance.")
print("Compare F1-scores (not shown here) for a fairer evaluation.")

---
## Task 4 – Feature Selection with `SelectFromModel`

In [ ]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])

model_fs = Pipeline([
    ("preprocess", preprocess),
    ("select", SelectFromModel(
        estimator=RandomForestClassifier(n_estimators=300, random_state=42,
                                         n_jobs=-1, class_weight="balanced"),
        threshold="median"
    )),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=42,
                                   n_jobs=-1, class_weight="balanced"))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)
acc_fs = accuracy_score(y_test, y_pred_fs)
n_selected = model_fs.named_steps["select"].get_support().sum()

print(f"Features selected : {n_selected}")
print(f"Baseline accuracy : {acc_base:.4f}")
print(f"After selection   : {acc_fs:.4f}")
print(classification_report(y_test, y_pred_fs, zero_division=0))
print()
print("Explanation:")
print("  SelectFromModel keeps only features above the median importance score,")
print("  cutting roughly half the features. For student data, weak demographic")
print("  features (e.g. nursery, paid) are likely pruned while strong predictors")
print("  (failures, studytime, absences) are retained. If accuracy is unchanged,")
print("  feature selection is beneficial — fewer features, same performance.")